In [1]:
import kagglehub
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split

In [3]:
path = kagglehub.dataset_download("shanegerami/ai-vs-human-text")

print("Path to dataset files:", path)
os.listdir(path)

Path to dataset files: C:\Users\osten\.cache\kagglehub\datasets\shanegerami\ai-vs-human-text\versions\1


['AI_Human.csv']

In [4]:
df = pd.read_csv(os.path.join(path, "AI_Human.csv"))
df

,text,generated
0,Cars. Cars have been around since they became ...,0.0
1,Transportation is a large necessity in most co...,0.0
2,"""America's love affair with it's vehicles seem...",0.0
3,How often do you ride in a car? Do you drive a...,0.0
4,Cars are a wonderful thing. They are perhaps o...,0.0
...,...,...
487230,Tie Face on Mars is really just a big misunder...,0.0
487231,The whole purpose of democracy is to create a ...,0.0
487232,I firmly believe that governments worldwide sh...,1.0
487233,I DFN't agree with this decision because a LFT...,0.0


In [21]:
pd.set_option('display.max_colwidth', 30)

In [ ]:
df_lower = df.copy()

#Lowercasing
for col in df.select_dtypes(include=['object']).columns:
        df_lower[col] = df_lower[col].str.lower()

#Noise Removal
def remove_noise(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z0-9.\s]', '', text)
    return text

df_lower['text'] = df_lower['text'].apply(remove_noise)

# Tokenization
df_lower['tokenized_text'] = df_lower['text'].apply(lambda x: x.split())

#Filtering
min_text_length = 30
df_filtered = df_lower[df_lower['tokenized_text'].apply(len) >= min_text_length].copy()

df_filtered

In [5]:
df_filtered['tokenized_text'].apply(len).min()

32

In [ ]:
df_preprocessed = df_filtered

In [7]:
df_preprocessed['generated'].value_counts()

generated
0.0    305796
1.0    181391
Name: count, dtype: int64

In [ ]:
df_ai = df_preprocessed[df_preprocessed['generated']==1]
df_human = df_preprocessed[df_preprocessed['generated']==0]

In [ ]:
df_human = df_human.sample(n=len(df_ai), random_state=3)

In [25]:
len(df_human)

181391

In [26]:
df_preprocessed=pd.concat([df_human,df_ai],ignore_index=True)

In [27]:
df_preprocessed['generated'].value_counts()

generated
0.0    181391
1.0    181391
Name: count, dtype: int64

In [28]:
df_preprocessed.to_csv('downsampled.csv',index=False)

# Feature engineering

In [3]:
pip install nltk

  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached regex-2025.11.3-cp311-cp311-win_amd64.whl.metadata (41 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached regex-2025.11.3-cp311-cp311-win_amd64.whl (277 kB)

   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   ---------------------------------------- 2/2 [nltk]

Note: you may need to restart the kernel to use updated packages.


In [4]:
from collections import Counter
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
from scipy.sparse import hstack, csr_matrix, save_npz
import numpy as np

In [5]:
df_preprocessed = pd.read_csv('downsampled.csv')

In [6]:
train_df, test_df = train_test_split(df_preprocessed, test_size=0.2, random_state=3, stratify=df_preprocessed['generated'])

In [7]:
#Entropy
def calculate_entropy(text_tokens):
    if not text_tokens:
        return 0.0
    counts = Counter(text_tokens)
    probabilities = [count / len(text_tokens) for count in counts.values()]
    entropy = -sum(p * math.log2(p) for p in probabilities)
    return entropy

train_df['entropy'] = train_df['tokenized_text'].apply(calculate_entropy)
test_df['entropy'] = test_df['tokenized_text'].apply(calculate_entropy)
df_preprocessed['entropy'] = df_preprocessed['tokenized_text'].apply(calculate_entropy)

In [8]:
# Burstiness
def calculate_burstiness(text):
    if not isinstance(text, str):
        return 0.0 # Handle non-string input
    sentences = sent_tokenize(text)
    if len(sentences) < 2:
        return 0.0 # Need at least 2 sentences to calculate variance
    sentence_lengths = [len(s.split()) for s in sentences]
    # Avoid division by zero if all sentence lengths are the same
    if len(set(sentence_lengths)) == 1:
        return 0.0
    return pd.Series(sentence_lengths).var()

train_df['burstiness'] = train_df['text'].apply(calculate_burstiness)
test_df['burstiness'] = test_df['text'].apply(calculate_burstiness)
df_preprocessed['burstiness'] = df_preprocessed['text'].apply(calculate_burstiness)

In [9]:
# Lexical Diversity
def calculate_lexical_diversity(text_tokens):
    if not text_tokens:
        return 0.0
    return len(set(text_tokens)) / len(text_tokens)

train_df['lexical_diversity'] = train_df['tokenized_text'].apply(calculate_lexical_diversity)
test_df['lexical_diversity'] = test_df['tokenized_text'].apply(calculate_lexical_diversity)
df_preprocessed['lexical_diversity'] = df_preprocessed['tokenized_text'].apply(calculate_lexical_diversity)

In [10]:
# TF-IDF Vector
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_train = vectorizer.fit_transform(train_df['text'])
tfidf_test = vectorizer.transform(test_df['text'])
tfidf = vectorizer.transform(df_preprocessed['text'])

In [11]:
tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 55470281 stored elements and shape (362782, 5000)>

In [12]:
numeric_features = ['entropy', 'burstiness', 'lexical_diversity']
X_numeric = df_preprocessed[numeric_features].values

X = hstack([tfidf, csr_matrix(X_numeric)])

In [14]:
df = pd.DataFrame.sparse.from_spmatrix(X)

In [ ]:
df.to_csv('preprocessed_dataset.csv',index=False)

In [25]:
train_df.head()

,text,generated,tokenized_text,entropy,burstiness,lexical_diversity
123581,in the passage the challenge of exploring venu...,0.0,"['in', 'the', 'passage', 'the', 'challenge', '...",3.875388,41.515152,0.014514
358749,there is nb single answer tb the question bf w...,1.0,"['there', 'is', 'nb', 'single', 'answer', 'tb'...",3.953455,22.125000,0.012462
325319,here is a possible essay\n\nthere are many ad...,1.0,"['here', 'is', 'a', 'possible', 'essay', 'ther...",3.922564,21.356061,0.018123
327199,wsrk together ts improve peoples lives.\n\nthe...,1.0,"['wsrk', 'together', 'ts', 'improve', 'peoples...",3.987329,45.914286,0.009230
139732,teachername\n\nschoolname\n\ndes plaines illin...,0.0,"['teachername', 'schoolname', 'des', 'plaines'...",3.850924,73.098522,0.008522


In [26]:
train_df['generated'][0]

np.float64(0.0)

In [27]:
numeric_features = ['entropy', 'burstiness', 'lexical_diversity']
X_train_numeric = train_df[numeric_features].values
X_test_numeric = test_df[numeric_features].values

X_train = hstack([tfidf_train, csr_matrix(X_train_numeric)])
X_test = hstack([tfidf_test, csr_matrix(X_test_numeric)])

Y_train = train_df['generated'].values
Y_test = test_df['generated'].values

In [28]:
save_npz("X_train.npz", X_train)
save_npz("X_test.npz", X_test)

np.save("Y_train.npy", Y_train)
np.save("Y_test.npy", Y_test)

In [29]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 45235454 stored elements and shape (290225, 5003)>